# Packages import and installation

In [0]:
%pip install ..

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import os

from ag_vision.pipelines import image_processing as ip
from ag_vision.pipelines import dataframe_processing as dfp

# Variables set up

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
# Images table name
table_name = "images_draft"

In [0]:
catalog_name = dbutils.secrets.get("enablement-package", "catalog_name")
imgs_df = spark.table(f"{catalog_name}.production.{table_name}").toPandas()
print(len(imgs_df))

In [0]:
# Set to maximum number of potentially recent images with no metadata files
n_recent_max = 15000

# Table entries retrieval and potential metadata recording

In [0]:
imgs_df['upload_date_parsed'] = pd.to_datetime(imgs_df['upload_date'], errors='coerce')
recent_imgs_df = imgs_df.sort_values('upload_date_parsed', ascending=False).head(n_recent_max)
print(len(recent_imgs_df))

In [0]:
missing_metadata_imgs_df = dfp.keep_records_with_missing_metadata_files(recent_imgs_df)
n_missing_metadata_imgs = len(missing_metadata_imgs_df)

if n_missing_metadata_imgs == 0:
    print("All of the recently added images have metadata files.")
else:
    print("The number of recently added images without metadata files is: ", n_missing_metadata_imgs)

In [0]:
# enerate metadata files for recently added images if no associated metadata files found
if n_missing_metadata_imgs:
    ip.generate_metadata_files_from_image_list(file_paths=missing_metadata_imgs_df['file_path'].tolist(),
                                               platform='db',
                                               image_type='original')